# Insulin Receptor — Small-Molecule Virtual Screening

Dock several candidate inhibitors into **one** insulin-receptor kinase structure and rank them
by predicted binding affinity — a virtual-screening "leaderboard" — then validate by redocking
the complex's own native ligand and measuring RMSD to its crystal pose.

- **Receptor:** pick one of four INSR kinase complexes in `data/insr/complexes/` (set `RECEPTOR`).
- **Candidates:** shared small-molecule library in `data/insr/candidates/`.
- **Engine:** AutoDock-GPU on a GPU node, else AutoDock Vina — chosen automatically.

Everything loads from the repo's own `data/` folder, so it runs the same on a laptop and on
Perlmutter (no internet needed).

In [ ]:
# --- Setup: make daladock importable and set the working directory ---
%load_ext autoreload
%autoreload 2
import sys, os, pathlib

# Clean PYTHONPATH to avoid conflicts
for _p in os.environ.get('PYTHONPATH', '').split(os.pathsep):
    while _p and _p in sys.path:
        sys.path.remove(_p)

# Locate the repo root (the folder containing src/daladock) — works on Mac and Perlmutter
_here = pathlib.Path.cwd()
REPO = next((c for c in [_here, *_here.parents] if (c / 'src' / 'daladock').is_dir()), _here)
sys.path.insert(0, str(REPO / 'src'))

WORK = REPO / 'work'
WORK.mkdir(exist_ok=True)
os.chdir(WORK)                 # all outputs go here
DATA = REPO / 'data'

import pandas as pd
import matplotlib.pyplot as plt
from daladock import prep_receptor, box, prep_ligand, dock, analyze, viz

print('repo        :', REPO)
print('working dir :', WORK)
print('docking engine:', dock.detect_engine())

## Stage 1 — Prepare the receptor

Set `RECEPTOR` to your assigned complex. `prepare_receptor(..., strip_hetero=True)` cleans the
protein *and* auto-extracts its co-crystal ligand as a reference (used to place the box).
`NATIVE` is that complex's own ligand — the ground truth for the Stage 8 validation.

In [ ]:
RECEPTOR = '5E1S'   # <-- set your assigned complex:  1IR3 | 3EKN | 4IBM | 5E1S

rec = prep_receptor.prepare_receptor(
    str(DATA / 'insr' / 'complexes' / f'{RECEPTOR}.pdb'),
    out=RECEPTOR, strip_hetero=True)          # receptor + auto-extracted native ligand (ref)

NATIVE = {'1IR3': 'AMP-PNP', '3EKN': 'GS3',
          '4IBM': 'Irfin-1', '5E1S': 'BI-885578'}[RECEPTOR]   # this complex's own ligand

print('receptor       :', rec['pdbqt'])
print('reference ligand:', rec['ref_ligand'])
print('native ligand   :', NATIVE)
viz.view_structure(rec['pdbqt'])

## Stage 2 — Define the search box

If you set **both** `CENTER` and `SIZE`, the box is placed exactly there. Otherwise it falls
back to `DETECT` (default `'ref'` = center + size from the co-crystal ligand = the ATP pocket).

In [ ]:
# --- Define the search box (works for ANY molecule, small or large) ---
# DETECT modes (used only when CENTER+SIZE are not both set):
#   'ref'     -> center + size from a reference ligand bound in the site (set REF)
#   'fpocket' -> auto-detect pockets, use top-ranked (VERIFY it's the right site!)
#   'blind'   -> box encloses the WHOLE receptor (any ligand; slower, less precise)
#   'auto'    -> ref if REF given, else fpocket, else blind

DETECT = 'ref'                  # fallback: 'ref' | 'fpocket' | 'blind' | 'auto'
CENTER = None                   # e.g. [x, y, z] to force the box center
SIZE   = None                   # e.g. [22, 22, 22] small molecule / [40, 40, 40] peptide
REF    = rec['ref_ligand']      # this complex's co-crystal ligand -> centers on the ATP pocket
PAD    = 8.0                    # margin around the ligand/pocket, Angstrom

if CENTER is not None and SIZE is not None:
    b = box.define_box(rec['pdbqt'], center=CENTER, size=SIZE)   # manual override
else:
    b = box.define_box(rec['pdbqt'], ref=REF, detect=DETECT, pad=PAD)

print('box center:', b['center'], ' size:', b['size'])
viz.view_box(rec['pdbqt'], b['center'], b['size'])

## Stage 3 — Prepare the candidate ligands (from PDB files)

The shared candidate library lives in `data/insr/candidates/`. Small-molecule PDBs are
bond-perceived automatically (via OpenBabel). You can also add your own `.pdb`/`.sdf`/SMILES.

In [ ]:
candidate_files = {n: str(DATA / 'insr' / 'candidates' / f'{n}.pdb')
                   for n in ['AMP-PNP', 'GS3', 'Irfin-1', 'BI-885578']}

ligands = {}
for name, pdb in candidate_files.items():
    if not os.path.exists(pdb):
        print(f'WARNING: {pdb} not found -- skipping {name}'); continue
    ligands[name] = prep_ligand.prepare_ligand(pdb, out=name)
    print(f"  {name}: {ligands[name]['n_torsions']} torsions, types={ligands[name]['types']}")

viz.view_ligand(ligands[list(ligands)[0]]['pdbqt'])

## Increasing the resource limit to avoid crash due to workload

In [ ]:
import resource
resource.setrlimit(resource.RLIMIT_STACK, (resource.RLIM_INFINITY, resource.RLIM_INFINITY))
print("stack limit:", resource.getrlimit(resource.RLIMIT_STACK))

## Stage 4 — Dock every candidate into the SAME receptor

The "one receptor, many candidates" screen: identical receptor and box for all, so scores are
comparable. `engine='auto'` uses AutoDock-GPU if a GPU is present (set `$ADGPU`), else Vina.

In [ ]:
results = {}
for name, lig in ligands.items():
    print(f'--- Docking {name} ---')
    results[name] = dock.dock(
        rec['pdbqt'], lig['pdbqt'], b['center'], b['size'],
        engine='auto', out=name, n_poses=10, exhaustiveness=8)
    print(f"  best score: {results[name]['best_score']:.2f} kcal/mol")

## Stage 5 — Rank the candidates (leaderboard)

Lower (more negative) `best_score_kcal_mol` = stronger predicted binding. Sanity check: the
receptor's own native ligand should rank at or near the top.

In [ ]:
board = analyze.leaderboard(results, csv='insr_leaderboard.csv')
display(board)

plt.figure(figsize=(6, 3))
plt.bar(board['ligand'], board['best_score_kcal_mol'], color='steelblue')
plt.ylabel('Best affinity (kcal/mol)')
plt.title(f'{RECEPTOR}: candidate ranking')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

top = board.iloc[0]['ligand']
print('Top-ranked:', top, '| Native ligand:', NATIVE,
      '->  CORRECT CALL!' if top == NATIVE else '->  native did not rank #1')

## Stage 6 — Visualize the top-ranked pose

In [ ]:
best = board.iloc[0]['ligand']
print(f"Top candidate: {best}  ({board.iloc[0]['best_score_kcal_mol']:.2f} kcal/mol)")
viz.view_complex(rec['pdbqt'], results[best]['poses'])

## Stage 7 — Validation: does docking recover a known pose?

This complex was solved **with `NATIVE` bound**, so its crystal position is the ground truth.
We compare the redocked native pose to the crystal: **RMSD < 2 A = docking is trustworthy.**

In [ ]:
# RMSD: docked top pose of the native ligand vs its crystal (input) pose.
_, ref_c  = analyze._parse_poses(ligands[NATIVE]['pdbqt'])    # crystal pose
_, dock_c = analyze._parse_poses(results[NATIVE]['poses'])    # docked poses
if ref_c and dock_c and ref_c[0].shape == dock_c[0].shape:
    rmsd = analyze._rmsd(dock_c[0], ref_c[0])
    print(f'{NATIVE}: top-pose RMSD to crystal = {rmsd:.2f} A',
          '-> PASS' if rmsd < 2.0 else '-> inspect the overlay')
else:
    print('atom counts differ - rely on the visual overlay below')

# Overlay: docked pose (orange) vs crystal ligand (green)
viz.view_pose_vs_reference(rec['pdbqt'], results[NATIVE]['poses'], candidate_files[NATIVE])

### Notes

- **Switch complex:** change `RECEPTOR` in Stage 1 (`1IR3` / `3EKN` / `4IBM` / `5E1S`) — everything
  else adapts automatically (box, native ligand, validation).
- **Engine-agnostic:** the same code runs on Vina (CPU) and AutoDock-GPU (set `$ADGPU`).
- **Add candidates:** drop more `.pdb`/`.sdf` into `data/insr/candidates/` and list them in Stage 3.
- **Scores are approximate** — compare trends, not absolute numbers.